In [1]:
import sys
sys.path.append("..")

In [2]:
from src import models

device = "cuda"
mt = models.load_model("gptj", device=device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/285 [00:00<?, ?it/s]

[transformers] GPTJForCausalLM LOAD REPORT from: EleutherAI/gpt-j-6B
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...27}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...27}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
from src import data

dataset = data.load_dataset()

In [4]:
from src import operators

estimator = operators.JacobianIclMeanEstimator(mt=mt, h_layer=11, z_layer=26)
relation = dataset[0].set(
    samples=dataset[0].samples[:3],
    prompt_templates=[dataset[0].prompt_templates[0]],
)
operator = estimator(relation)

In [5]:
operator("Italy").predictions

[PredictedToken(token=' men', prob=0.7421227693557739),
 PredictedToken(token=' women', prob=0.22282566130161285),
 PredictedToken(token='\n', prob=0.011809328570961952),
 PredictedToken(token=' the', prob=0.00492286030203104),
 PredictedToken(token=' people', prob=0.0021172957494854927)]

In [6]:
from src import functional

import baukit
import torch

def apply_lm_head(hiddens):
    predictions = mt.lm_head[1:](hiddens).topk(dim=-1, k=10).indices
    return [
        [mt.tokenizer.decode(t) for t in ts]
        for ts in predictions
    ]


outputs = functional.compute_hidden_states(mt=mt, layers=[26], prompt=[
    "The capital of Norway is",
    "The capital of France is",
])
z = outputs.hiddens[0][0, -1]
z_old = outputs.hiddens[0][-1, -1]
print(z)
print(z_old)

# j_inv = torch.linalg.pinv(operator.weight.float()).half()
j_inv = torch.eye(4096).half().to(device)

delta = j_inv.mm(z[..., None] - operator.bias.t())
print(delta.norm())

delta_old = j_inv.mm(z_old[..., None] - operator.bias.t())
print(delta_old.norm())

def edit_output(output, layer):
    if output[0].shape[1] == 1:
        return output
    print(output[0][0, 3].norm())
    output[0][0, 3] = (
#         -1 * delta_old.squeeze() +
        delta.squeeze()
    )
    print(output[0][0, 3].norm())
    return output


inputs = mt.tokenizer("The capital of France is the city of", return_tensors="pt").to(device)
with baukit.TraceDict(mt.model, [f"transformer.h.{l}" for l in [11]], edit_output=edit_output):
    outputs = mt.model.generate(
        **inputs,
        pad_token_id=mt.tokenizer.eos_token_id,
        max_new_tokens=100,
    )
mt.tokenizer.batch_decode(outputs)[0]

tensor([-4.2305, -0.9780, -2.1074,  ..., -0.1805, -0.7329, -2.7930],
       device='cuda:0', dtype=torch.float16)
tensor([-3.5645, -1.8535, -1.5293,  ...,  1.3926, -3.1680, -1.6748],
       device='cuda:0', dtype=torch.float16)
tensor(214.2500, device='cuda:0', dtype=torch.float16,
       grad_fn=<LinalgVectorNormBackward0>)
tensor(216.6250, device='cuda:0', dtype=torch.float16,
       grad_fn=<LinalgVectorNormBackward0>)
tensor(114.2500, device='cuda:0', dtype=torch.float16)
tensor(214.2500, device='cuda:0', dtype=torch.float16)


'The capital of France is the city of Rome. The capital of the United States is the city of Washington. The capital of the United Kingdom is the city of London. The capital of Canada is the city of Ottawa. The capital of Australia is the city of Canberra. The capital of New Zealand is the city of Wellington. The capital of the Philippines is the city of Manila. The capital of Indonesia is the city of Jakarta. The capital of Malaysia is the city of Kuala Lumpur. The capital of Thailand is the city of Bangkok. The'

In [7]:
import torch

def apply_lm_head(hiddens):
    predictions = mt.lm_head[1:](hiddens).argmax(dim=-1)
    print(predictions.shape)
    return [(i, mt.tokenizer.decode(tok)) for i, tok in enumerate(predictions.tolist())]


@torch.inference_mode()
def logit_lens(prompt):
    inputs = mt.tokenizer(prompt, return_tensors="pt", padding="longest", truncation=True).to(device)
    inputs.pop("token_type_ids", None)
    outputs = mt.model(**inputs, return_dict=True, output_hidden_states=True)
    hiddens = torch.cat(outputs.hidden_states[1:], dim=0)[:, -1]
    return apply_lm_head(hiddens)


logit_lens("The capital of France is")

torch.Size([28])


[(0, ' also'),
 (1, ' still'),
 (2, ' still'),
 (3, ' still'),
 (4, ' still'),
 (5, ' often'),
 (6, ' often'),
 (7, ' still'),
 (8, ' still'),
 (9, ' still'),
 (10, ' city'),
 (11, ' also'),
 (12, ' Paris'),
 (13, ' city'),
 (14, ' London'),
 (15, ' not'),
 (16, ' Paris'),
 (17, ' Paris'),
 (18, ' Paris'),
 (19, ' Paris'),
 (20, ' Paris'),
 (21, ' Paris'),
 (22, ' Paris'),
 (23, ' Paris'),
 (24, ' Paris'),
 (25, ' a'),
 (26, ' a'),
 (27, ' a')]